In [1]:
# getting file

!pip -q install kaggle

from google.colab import files
from pathlib import Path
import pandas as pd
import numpy as np
import random
import os

uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle configured.")

Saving kaggle.json to kaggle.json
Kaggle configured.


In [2]:
PROJECT_DIR = Path("/content/MammoGraph")
DATASET_DIR = PROJECT_DIR / "dataset"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Dataset:", DATASET_DIR)

Project: /content/MammoGraph
Dataset: /content/MammoGraph/dataset


In [3]:
!kaggle datasets download \
    -d shantanughosh/vindr-mammogram-dataset-dicom-to-png \
    -p /content/MammoGraph/dataset

Dataset URL: https://www.kaggle.com/datasets/shantanughosh/vindr-mammogram-dataset-dicom-to-png
License(s): CC-BY-NC-SA-4.0
100% 7.68G/7.68G [01:15<00:00, 109MB/s] 



In [4]:
!ls -lh /content/MammoGraph/dataset/

!unzip -l \
    /content/MammoGraph/dataset/vindr-mammogram-dataset-dicom-to-png.zip | head -30

total 7.7G
-rw-r--r-- 1 root root 7.7G Jun 23  2024 vindr-mammogram-dataset-dicom-to-png.zip
Archive:  /content/MammoGraph/dataset/vindr-mammogram-dataset-dicom-to-png.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   442407  2024-06-23 03:23   images_png/0025a5dc99fd5c742026f0b2b030d3e9/2ddfad7286c2b016931ceccd1e2c7bbc.png
   366322  2024-06-23 03:23   images_png/0025a5dc99fd5c742026f0b2b030d3e9/451562831387e2822923204cf8f0873e.png
   444199  2024-06-23 03:23   images_png/0025a5dc99fd5c742026f0b2b030d3e9/47c8858666bcce92bcbd57974b5ce522.png
   367476  2024-06-23 03:23   images_png/0025a5dc99fd5c742026f0b2b030d3e9/fcf12c2803ba8dc564bf1287c0c97d9a.png
   449664  2024-06-23 03:23   images_png/0028fb2c7f0b3a5cb9a80cb0e1cdbb91/16e58fc1d65fa7587247e6224ee96527.png
   392151  2024-06-23 03:23   images_png/0028fb2c7f0b3a5cb9a80cb0e1cdbb91/3704f91985dcbc69f6ac2803523d1ecb.png
   485113  2024-06-23 03:23   images_png/0028fb2c7f0b3a5cb9a80cb0e1cdbb91/7fc1f1bb8bb1a7efaf

In [5]:
!unzip -q \
    /content/MammoGraph/dataset/vindr-mammogram-dataset-dicom-to-png.zip \
    -d /content/MammoGraph/dataset/

print("Extraction complete.")

Extraction complete.


In [6]:
image_dir = DATASET_DIR / "images_png"

study_dirs = [
    p for p in image_dir.iterdir()
    if p.is_dir()
]

print("Study folders:", len(study_dirs))

if study_dirs:

    first_study = study_dirs[0]

    images = list(
        first_study.glob("*.png")
    )

    print(
        "Example study:",
        first_study.name
    )

    print(
        "Images in example study:",
        len(images)
    )

    for img in images:
        print(" ", img.name)

Study folders: 5000
Example study: a4d42bf9c1134be1dc77999efe553c97
Images in example study: 4
  fb04090d40d2949cff34cd0e1726a3df.png
  3cdef45e87af9ce95fed666fa8386c18.png
  97bc5e0910de874eec16300587775067.png
  2c1a6a97af8a309d61a5ec562afa3393.png


In [7]:
!rm /content/MammoGraph/dataset/vindr-mammogram-dataset-dicom-to-png.zip

!du -sh /content/MammoGraph/dataset

7.8G	/content/MammoGraph/dataset


In [8]:
!git clone -q \
    https://github.com/batmanlab/Mammo-CLIP.git \
    /content/Mammo-CLIP

print("Mammo-CLIP cloned.")

Mammo-CLIP cloned.


In [9]:
!find /content/Mammo-CLIP \
    -name "vindr_detection_v1_folds.csv"

/content/Mammo-CLIP/src/codebase/data_csv/vindr_detection_v1_folds.csv


In [10]:
!cp \
    /content/Mammo-CLIP/src/codebase/data_csv/vindr_detection_v1_folds.csv \
    /content/MammoGraph/dataset/

In [11]:
csv_path = (
    DATASET_DIR /
    "vindr_detection_v1_folds.csv"
)

print("Exists:", csv_path.exists())

print(
    "Size:",
    round(
        csv_path.stat().st_size / 1024 / 1024,
        2
    ),
    "MB"
)

df = pd.read_csv(csv_path)

print("CSV shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Exists: True
Size: 4.07 MB
CSV shape: (20486, 33)

Columns:
['patient_id', 'series_id', 'image_id', 'laterality', 'view', 'height', 'width', 'breast_birads', 'breast_density', 'finding_categories', 'finding_birads', 'xmin', 'ymin', 'xmax', 'ymax', 'split', 'resized_xmin', 'resized_ymin', 'resized_xmax', 'resized_ymax', 'fold', 'Architectural_Distortion', 'Asymmetry', 'Focal_Asymmetry', 'Global_Asymmetry', 'Mass', 'Nipple_Retraction', 'No_Finding', 'Skin_Retraction', 'Skin_Thickening', 'Suspicious_Calcification', 'Suspicious_Lymph_Node', 'density']


/tmp/ipykernel_4518/526008954.py:17: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


In [12]:
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# Official VinDr training studies
# ------------------------------------------------------------

training_studies = (
    df.loc[
        df["split"] == "training",
        "patient_id"
    ]
    .drop_duplicates()
    .tolist()
)

print(
    "Official training studies:",
    len(training_studies)
)

# ------------------------------------------------------------
# 80/20 study-level train/validation split
# ------------------------------------------------------------

train_studies, val_studies = train_test_split(
    training_studies,
    test_size=0.20,
    random_state=42
)

train_studies = set(train_studies)
val_studies = set(val_studies)

print(
    "Training studies:",
    len(train_studies)
)

print(
    "Validation studies:",
    len(val_studies)
)

# ------------------------------------------------------------
# Build dataframes
# ------------------------------------------------------------

train_df = df[
    (df["split"] == "training") &
    (df["patient_id"].isin(train_studies))
].copy()

val_df = df[
    (df["split"] == "training") &
    (df["patient_id"].isin(val_studies))
].copy()

test_df = df[
    df["split"] == "test"
].copy()

print("\nRows:")
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

Official training studies: 4000
Training studies: 3200
Validation studies: 800

Rows:
Train: 13101
Val  : 3290
Test : 4095


In [13]:
SPLIT_DIR = DATASET_DIR / "splits"

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

train_df.to_csv(
    SPLIT_DIR / "train.csv",
    index=False
)

val_df.to_csv(
    SPLIT_DIR / "val.csv",
    index=False
)

test_df.to_csv(
    SPLIT_DIR / "test.csv",
    index=False
)

print("Splits saved.")

print(
    list(SPLIT_DIR.iterdir())
)

Splits saved.
[PosixPath('/content/MammoGraph/dataset/splits/val.csv'), PosixPath('/content/MammoGraph/dataset/splits/train.csv'), PosixPath('/content/MammoGraph/dataset/splits/test.csv')]


In [14]:
train_ids = set(
    train_df["patient_id"].unique()
)

val_ids = set(
    val_df["patient_id"].unique()
)

test_ids = set(
    test_df["patient_id"].unique()
)

print(
    "Train ∩ Val:",
    len(train_ids & val_ids)
)

print(
    "Train ∩ Test:",
    len(train_ids & test_ids)
)

print(
    "Val ∩ Test:",
    len(val_ids & test_ids)
)

Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


GNT TRAINING

In [11]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import models, transforms
from PIL import Image

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DATASET_DIR = Path("/content/MammoGraph/dataset")
IMAGE_DIR = DATASET_DIR / "images_png"

FINDING_CLASSES = [
    "Mass",
    "Suspicious Calcification",
    "Focal Asymmetry",
    "Architectural Distortion",
    "Asymmetry",
    "Suspicious Lymph Node",
    "Skin Thickening",
    "Global Asymmetry",
    "Nipple Retraction",
    "Skin Retraction",
]

NUM_CLASSES = len(FINDING_CLASSES)

IMG_WIDTH = 912
IMG_HEIGHT = 1520

print("Classes:", NUM_CLASSES)

Device: cuda
GPU: Tesla T4
Classes: 10


In [12]:
csv_path = DATASET_DIR / "vindr_detection_v1_folds.csv"

df = pd.read_csv(csv_path)

print("Full CSV:", df.shape)

train_df = pd.read_csv(DATASET_DIR / "splits/train.csv")
val_df   = pd.read_csv(DATASET_DIR / "splits/val.csv")
test_df  = pd.read_csv(DATASET_DIR / "splits/test.csv")

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

Full CSV: (20486, 33)
Train: (13101, 33)
Val  : (3290, 33)
Test : (4095, 33)


/tmp/ipykernel_74375/129085418.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


In [13]:
class MammoGNTDataset(Dataset):

    def __init__(self, dataframe, image_dir):
        self.df = dataframe.copy()
        self.image_dir = Path(image_dir)

        # Each unique image gets one dataset item.
        self.image_ids = self.df["image_id"].unique().tolist()

        # Group annotations by image
        self.annotations = {
            image_id: group
            for image_id, group in self.df.groupby("image_id")
        }

    def __len__(self):
        return len(self.image_ids)

    def _parse_findings(self, value):

        if pd.isna(value):
            return []

        return [
            x.strip()
            for x in str(value).split(",")
            if x.strip() in FINDING_CLASSES
        ]

    def __getitem__(self, idx):

        image_id = self.image_ids[idx]

        rows = self.annotations[image_id]

        # patient_id is the study folder
        study_id = str(rows.iloc[0]["patient_id"])

        image_path = self.image_dir / study_id / image_id

        image = Image.open(image_path).convert("L")

        # Keep native 912 x 1520 resolution
        image = torch.from_numpy(
            np.array(image, dtype=np.float32)
        ) / 255.0

        image = image.unsqueeze(0)

        image_labels = torch.zeros(NUM_CLASSES, dtype=torch.float32)

        boxes = []
        box_labels = []

        for _, row in rows.iterrows():

            findings = self._parse_findings(
                row["finding_categories"]
            )

            # Image-level multilabel target
            for finding in findings:
                image_labels[
                    FINDING_CLASSES.index(finding)
                ] = 1.0

            # Only rows with actual bounding boxes
            if (
                pd.notna(row["resized_xmin"])
                and pd.notna(row["resized_ymin"])
                and pd.notna(row["resized_xmax"])
                and pd.notna(row["resized_ymax"])
            ):

                xmin = float(row["resized_xmin"])
                ymin = float(row["resized_ymin"])
                xmax = float(row["resized_xmax"])
                ymax = float(row["resized_ymax"])

                # Clip to PNG boundaries
                xmin = max(0.0, min(xmin, IMG_WIDTH))
                xmax = max(0.0, min(xmax, IMG_WIDTH))

                ymin = max(0.0, min(ymin, IMG_HEIGHT))
                ymax = max(0.0, min(ymax, IMG_HEIGHT))

                if xmax > xmin and ymax > ymin:

                    boxes.append([
                        xmin,
                        ymin,
                        xmax,
                        ymax
                    ])

                    labels = torch.zeros(
                        NUM_CLASSES,
                        dtype=torch.float32
                    )

                    for finding in findings:
                        labels[
                            FINDING_CLASSES.index(finding)
                        ] = 1.0

                    box_labels.append(labels)

        if boxes:
            boxes = torch.tensor(
                boxes,
                dtype=torch.float32
            )

            box_labels = torch.stack(box_labels)

        else:
            boxes = torch.zeros(
                (0, 4),
                dtype=torch.float32
            )

            box_labels = torch.zeros(
                (0, NUM_CLASSES),
                dtype=torch.float32
            )

        return {
            "image": image,
            "image_labels": image_labels,
            "boxes": boxes,
            "box_labels": box_labels,
            "image_id": image_id,
            "study_id": study_id,
        }

In [14]:
import ast

def fixed_parse_findings(self, value):

    if pd.isna(value):
        return []

    value = str(value).strip()

    # CSV stores findings as strings like:
    # "['Mass']"
    # "['Suspicious Calcification', 'Mass']"

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return [
                str(x).strip()
                for x in parsed
                if str(x).strip() in FINDING_CLASSES
            ]

    except (ValueError, SyntaxError):
        pass

    # Fallback for ordinary comma-separated strings
    return [
        x.strip()
        for x in value.split(",")
        if x.strip() in FINDING_CLASSES
    ]


# Replace the old parser
MammoGNTDataset._parse_findings = fixed_parse_findings

print("Parser fixed.")

Parser fixed.


In [15]:
train_dataset = MammoGNTDataset(
    train_df,
    IMAGE_DIR
)

val_dataset = MammoGNTDataset(
    val_df,
    IMAGE_DIR
)

test_dataset = MammoGNTDataset(
    test_df,
    IMAGE_DIR
)

print("Train images:", len(train_dataset))
print("Val images  :", len(val_dataset))
print("Test images :", len(test_dataset))

Train images: 12800
Val images  : 3200
Test images : 4000


In [16]:
def gnt_collate_fn(batch):

    images = torch.stack([
        item["image"]
        for item in batch
    ])

    image_labels = torch.stack([
        item["image_labels"]
        for item in batch
    ])

    return {
        "images": images,
        "image_labels": image_labels,
        "boxes": [
            item["boxes"] for item in batch
        ],
        "box_labels": [
            item["box_labels"] for item in batch
        ],
        "image_ids": [
            item["image_id"] for item in batch
        ],
        "study_ids": [
            item["study_id"] for item in batch
        ],
    }

In [17]:
# Identify images containing at least one abnormal finding.

positive_flags = []

for image_id in train_dataset.image_ids:

    rows = train_dataset.annotations[image_id]

    positive = False

    for _, row in rows.iterrows():

        findings = train_dataset._parse_findings(
            row["finding_categories"]
        )

        if len(findings) > 0:
            positive = True
            break

    positive_flags.append(positive)

positive_flags = np.array(positive_flags)

num_positive = positive_flags.sum()
num_negative = len(positive_flags) - num_positive

print("Positive images:", num_positive)
print("Negative images:", num_negative)

weights = np.where(
    positive_flags,
    1.0 / max(num_positive, 1),
    1.0 / max(num_negative, 1)
)

sampler = WeightedRandomSampler(
    weights=torch.tensor(weights, dtype=torch.double),
    num_samples=len(weights),
    replacement=True
)

Positive images: 1111
Negative images: 11689


In [18]:
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True,
    collate_fn=gnt_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=gnt_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=gnt_collate_fn
)

print("Loaders ready.")

Loaders ready.


In [19]:
weights = models.ResNet18_Weights.DEFAULT

resnet = models.resnet18(weights=weights)

# Convert first layer from RGB → grayscale
old_conv = resnet.conv1

new_conv = nn.Conv2d(
    1,
    old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

with torch.no_grad():
    new_conv.weight[:] = old_conv.weight.mean(
        dim=1,
        keepdim=True
    )

resnet.conv1 = new_conv

# Remove classification head
feature_extractor = nn.Sequential(
    *list(resnet.children())[:-2]
)

# Freeze backbone for GNT v1
for param in feature_extractor.parameters():
    param.requires_grad = False

feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()

print("Feature extractor ready.")

Feature extractor ready.


In [20]:
def feature_map_to_nodes(feature_map):

    B, C, H, W = feature_map.shape

    # [B, C, H, W] → [B, H*W, C]
    nodes = feature_map.flatten(
        2
    ).transpose(1, 2)

    # Normalized spatial coordinates
    y = torch.linspace(
        0, 1, H,
        device=feature_map.device
    )

    x = torch.linspace(
        0, 1, W,
        device=feature_map.device
    )

    yy, xx = torch.meshgrid(
        y, x, indexing="ij"
    )

    coords = torch.stack(
        [xx, yy],
        dim=-1
    ).reshape(1, H * W, 2)

    coords = coords.expand(B, -1, -1)

    nodes = torch.cat(
        [nodes, coords],
        dim=-1
    )

    return nodes, coords

In [21]:
def build_spatial_knn_neighbors(coords, k=8):

    """
    coords: [B, N, 2]

    Returns:
        neighbors: [B, N, k]

    Each node receives its k nearest spatial neighbors.
    """

    B, N, _ = coords.shape

    distances = torch.cdist(
        coords,
        coords
    )

    # Exclude self
    distances = distances.masked_fill(
        torch.eye(
            N,
            device=coords.device,
            dtype=torch.bool
        ).unsqueeze(0),
        float("inf")
    )

    neighbors = torch.topk(
        distances,
        k=k,
        dim=-1,
        largest=False
    ).indices

    return neighbors

In [22]:
class GraphTransformerLayer(nn.Module):

    def __init__(
        self,
        dim=256,
        num_heads=8,
        dropout=0.1
    ):
        super().__init__()

        assert dim % num_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)

        self.out_proj = nn.Linear(dim, dim)

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x, neighbors):

        """
        x:
            [N, D]

        neighbors:
            [N, K]
        """

        N, D = x.shape
        K = neighbors.shape[1]

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # [N, H, Dh]
        q = q.view(
            N,
            self.num_heads,
            self.head_dim
        )

        k = k.view(
            N,
            self.num_heads,
            self.head_dim
        )

        v = v.view(
            N,
            self.num_heads,
            self.head_dim
        )

        # Gather neighbors
        k_neighbors = k[neighbors]
        v_neighbors = v[neighbors]

        # q: [N, H, Dh]
        # k_neighbors: [N, K, H, Dh]

        scores = (
            q.unsqueeze(1) * k_neighbors
        ).sum(dim=-1)

        scores = scores / (
            self.head_dim ** 0.5
        )

        # Normalize across each node's neighbors
        attention = F.softmax(
            scores,
            dim=1
        )

        aggregated = (
            attention.unsqueeze(-1)
            * v_neighbors
        ).sum(dim=1)

        aggregated = aggregated.reshape(
            N,
            D
        )

        x = self.norm1(
            x + self.out_proj(aggregated)
        )

        x = self.norm2(
            x + self.ffn(x)
        )

        return x

In [23]:
class MammoGNT(nn.Module):

    def __init__(
        self,
        input_dim=514,
        hidden_dim=256,
        num_classes=10,
        num_layers=2,
        num_heads=8,
        k=8
    ):
        super().__init__()

        self.k = k

        self.node_projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )

        self.gnt_layers = nn.ModuleList([
            GraphTransformerLayer(
                dim=hidden_dim,
                num_heads=num_heads
            )
            for _ in range(num_layers)
        ])

        self.pool_norm = nn.LayerNorm(hidden_dim)

        # Image-level classification
        self.classifier = nn.Linear(
            hidden_dim,
            num_classes
        )

        # Node objectness
        self.objectness_head = nn.Linear(
            hidden_dim,
            1
        )

        # Node finding classification
        self.node_classifier = nn.Linear(
            hidden_dim,
            num_classes
        )

        # Bounding box regression
        self.bbox_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 4),
            nn.Sigmoid()
        )

    def forward(
        self,
        feature_nodes,
        neighbors
    ):

        """
        feature_nodes:
            [B, N, 514]

        neighbors:
            [B, N, K]
        """

        B, N, _ = feature_nodes.shape

        x = self.node_projection(
            feature_nodes
        )

        for layer in self.gnt_layers:

            outputs = []

            for b in range(B):

                outputs.append(
                    layer(
                        x[b],
                        neighbors[b]
                    )
                )

            x = torch.stack(outputs)

        # Global image representation
        pooled = x.mean(dim=1)

        pooled = self.pool_norm(
            pooled
        )

        image_logits = self.classifier(
            pooled
        )

        objectness = self.objectness_head(
            x
        ).squeeze(-1)

        node_class_logits = self.node_classifier(
            x
        )

        bbox_predictions = self.bbox_head(
            x
        )

        return {
            "image_logits": image_logits,
            "node_features": x,
            "objectness": objectness,
            "node_class_logits": node_class_logits,
            "bbox_predictions": bbox_predictions,
        }

In [24]:
model = MammoGNT(
    input_dim=514,
    hidden_dim=256,
    num_classes=NUM_CLASSES,
    num_layers=2,
    num_heads=8,
    k=8
).to(DEVICE)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

Trainable parameters: 1784601


In [25]:
def get_node_coordinates(
    batch_size,
    H,
    W,
    device
):

    y = torch.linspace(
        0, 1, H,
        device=device
    )

    x = torch.linspace(
        0, 1, W,
        device=device
    )

    yy, xx = torch.meshgrid(
        y, x, indexing="ij"
    )

    coords = torch.stack(
        [xx, yy],
        dim=-1
    ).reshape(1, H * W, 2)

    return coords.expand(
        batch_size,
        -1,
        -1
    )

In [26]:
def assign_nodes_to_box(
    node_coords,
    box
):

    """
    node_coords: [N, 2], normalized x/y
    box: [4], pixel coordinates
    """

    xmin, ymin, xmax, ymax = box

    xmin /= IMG_WIDTH
    xmax /= IMG_WIDTH

    ymin /= IMG_HEIGHT
    ymax /= IMG_HEIGHT

    inside = (
        (node_coords[:, 0] >= xmin) &
        (node_coords[:, 0] <= xmax) &
        (node_coords[:, 1] >= ymin) &
        (node_coords[:, 1] <= ymax)
    )

    return inside

In [27]:
def build_gnt_targets(
    coords,
    boxes,
    box_labels
):

    """
    coords:
        [N, 2]

    boxes:
        [M, 4]

    box_labels:
        [M, NUM_CLASSES]
    """

    N = coords.shape[0]

    objectness = torch.zeros(
        N,
        device=coords.device
    )

    node_labels = torch.zeros(
        N,
        NUM_CLASSES,
        device=coords.device
    )

    bbox_targets = torch.zeros(
        N,
        4,
        device=coords.device
    )

    assigned = torch.zeros(
        N,
        dtype=torch.bool,
        device=coords.device
    )

    for box_idx in range(len(boxes)):

        box = boxes[box_idx].to(coords.device)
        labels = box_labels[box_idx].to(coords.device)

        inside = assign_nodes_to_box(
            coords,
            box
        )

        # If no feature node falls inside,
        # assign nearest node to box center.
        if not inside.any():

            center_x = (
                (box[0] + box[2]) / 2
            ) / IMG_WIDTH

            center_y = (
                (box[1] + box[3]) / 2
            ) / IMG_HEIGHT

            distances = (
                (coords[:, 0] - center_x) ** 2
                +
                (coords[:, 1] - center_y) ** 2
            )

            nearest = distances.argmin()

            inside[nearest] = True

        # Objectness
        objectness[inside] = 1.0

        # Multilabel finding supervision
        node_labels[inside] = torch.maximum(
            node_labels[inside],
            labels.unsqueeze(0)
        )

        # Normalize bbox to 0–1
        normalized_box = torch.tensor(
            [
                box[0] / IMG_WIDTH,
                box[1] / IMG_HEIGHT,
                box[2] / IMG_WIDTH,
                box[3] / IMG_HEIGHT
            ],
            device=coords.device
        )

        # Don't overwrite existing box targets
        # unless this node isn't assigned yet.
        new_nodes = inside & (~assigned)

        bbox_targets[new_nodes] = normalized_box

        assigned[inside] = True

    return (
        objectness,
        node_labels,
        bbox_targets
    )

In [28]:
OBJECTNESS_POS_WEIGHT = 10.0

bce_objectness = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [OBJECTNESS_POS_WEIGHT],
        device=DEVICE
    )
)

bce_classification = nn.BCEWithLogitsLoss()

smooth_l1 = nn.SmoothL1Loss(
    reduction="none"
)

In [29]:
def gnt_loss(
    outputs,
    image_labels,
    objectness_targets,
    node_class_targets,
    bbox_targets
):

    # --------------------------------------------------------
    # Image-level classification
    # --------------------------------------------------------

    loss_image = bce_classification(
        outputs["image_logits"],
        image_labels
    )

    # --------------------------------------------------------
    # Node objectness
    # --------------------------------------------------------

    loss_objectness = bce_objectness(
        outputs["objectness"].unsqueeze(-1),
        objectness_targets.unsqueeze(-1)
    )

    # --------------------------------------------------------
    # Node finding classification
    # --------------------------------------------------------

    loss_node_class = bce_classification(
        outputs["node_class_logits"],
        node_class_targets
    )

    # --------------------------------------------------------
    # Bounding box regression
    # Only positive nodes contribute
    # --------------------------------------------------------

    positive_mask = (
        objectness_targets > 0
    )

    if positive_mask.any():

        pred_boxes = outputs[
            "bbox_predictions"
        ][positive_mask]

        true_boxes = bbox_targets[
            positive_mask
        ]

        loss_bbox = smooth_l1(
            pred_boxes,
            true_boxes
        ).mean()

    else:

        loss_bbox = torch.tensor(
            0.0,
            device=DEVICE
        )

    total_loss = (
        loss_image
        + loss_objectness
        + loss_node_class
        + loss_bbox
    )

    return total_loss, {
        "image": loss_image.item(),
        "objectness": loss_objectness.item(),
        "node_class": loss_node_class.item(),
        "bbox": loss_bbox.item(),
    }

In [30]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [31]:
batch = next(iter(train_loader))

images = batch["images"].to(
    DEVICE,
    non_blocking=True
)

print("Images:", images.shape)

with torch.no_grad():

    feature_map = feature_extractor(
        images
    )

print("Feature map:", feature_map.shape)

nodes, coords = feature_map_to_nodes(
    feature_map
)

print("Nodes:", nodes.shape)
print("Coords:", coords.shape)

neighbors = build_spatial_knn_neighbors(
    coords,
    k=8
)

print("Neighbors:", neighbors.shape)

print(
    "Neighbor min:",
    neighbors.min().item()
)

print(
    "Neighbor max:",
    neighbors.max().item()
)

Images: torch.Size([4, 1, 1520, 912])
Feature map: torch.Size([4, 512, 48, 29])
Nodes: torch.Size([4, 1392, 514])
Coords: torch.Size([4, 1392, 2])
Neighbors: torch.Size([4, 1392, 8])
Neighbor min: 0
Neighbor max: 1391


In [32]:
model.eval()

with torch.no_grad():

    outputs = model(
        nodes.to(DEVICE),
        neighbors.to(DEVICE)
    )

print("Image logits:",
      outputs["image_logits"].shape)

print("Objectness:",
      outputs["objectness"].shape)

print("Node classes:",
      outputs["node_class_logits"].shape)

print("BBox predictions:",
      outputs["bbox_predictions"].shape)

Image logits: torch.Size([4, 10])
Objectness: torch.Size([4, 1392])
Node classes: torch.Size([4, 1392, 10])
BBox predictions: torch.Size([4, 1392, 4])


In [33]:
batch = next(iter(train_loader))

images = batch["images"].to(DEVICE)

with torch.no_grad():
    feature_map = feature_extractor(images)

nodes, coords = feature_map_to_nodes(feature_map)

print("Batch diagnostic:")

for b in range(images.shape[0]):

    obj_target, node_target, bbox_target = \
        build_gnt_targets(
            coords[b],
            batch["boxes"][b],
            batch["box_labels"][b]
        )

    print(
        f"Image {b}: "
        f"GT boxes={len(batch['boxes'][b])}, "
        f"positive nodes={obj_target.sum().item()}"
    )

Batch diagnostic:
Image 0: GT boxes=3, positive nodes=59.0
Image 1: GT boxes=1, positive nodes=408.0
Image 2: GT boxes=1, positive nodes=42.0
Image 3: GT boxes=2, positive nodes=167.0


In [44]:
model.train()

optimizer.zero_grad()

outputs = model(
    nodes.to(DEVICE),
    neighbors.to(DEVICE)
)

image_labels = batch[
    "image_labels"
].to(DEVICE)

objectness_targets = []
node_class_targets = []
bbox_targets = []

for b in range(images.shape[0]):

    obj, node_cls, bbox = build_gnt_targets(
        coords[b],
        batch["boxes"][b],
        batch["box_labels"][b]
    )

    objectness_targets.append(obj)
    node_class_targets.append(node_cls)
    bbox_targets.append(bbox)

objectness_targets = torch.stack(
    objectness_targets
)

node_class_targets = torch.stack(
    node_class_targets
)

bbox_targets = torch.stack(
    bbox_targets
)

loss, loss_dict = gnt_loss(
    outputs,
    image_labels,
    objectness_targets,
    node_class_targets,
    bbox_targets
)

print("Loss:", loss.item())
print(loss_dict)

loss.backward()

optimizer.step()

print("Backward + optimizer step successful.")

Loss: 3.696887969970703
{'image': 0.6548538208007812, 'objectness': 2.185762643814087, 'node_class': 0.7398788332939148, 'bbox': 0.11639261245727539}
Backward + optimizer step successful.


In [38]:
print("================================================")
print("DATASET DIAGNOSTIC")
print("================================================")

print("\nTrain dataframe shape:")
print(train_df.shape)

print("\nTrain dataframe columns:")
print(train_df.columns.tolist())

print("\nFinding categories:")
print(
    train_df["finding_categories"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nNumber of non-empty finding rows:")

non_empty = train_df["finding_categories"].notna() & (
    train_df["finding_categories"].astype(str).str.strip() != ""
)

print(non_empty.sum())

print("\nBounding box availability:")

bbox_cols = [
    "resized_xmin",
    "resized_ymin",
    "resized_xmax",
    "resized_ymax"
]

for col in bbox_cols:
    print(
        col,
        "non-null:",
        train_df[col].notna().sum()
    )

print("\nUnique training images:")
print(train_df["image_id"].nunique())

print("\nImages containing findings:")

positive_image_ids = []

for image_id in train_dataset.image_ids:

    rows = train_dataset.annotations[image_id]

    positive = False

    for _, row in rows.iterrows():

        findings = train_dataset._parse_findings(
            row["finding_categories"]
        )

        if len(findings) > 0:
            positive = True
            break

    if positive:
        positive_image_ids.append(image_id)

print(
    "Positive images:",
    len(positive_image_ids)
)

print(
    "Negative images:",
    len(train_dataset.image_ids)
    - len(positive_image_ids)
)

print("\nSampler statistics:")
print("Positive flags:", positive_flags.sum())
print("Negative flags:", (~positive_flags).sum())

DATASET DIAGNOSTIC

Train dataframe shape:
(13101, 33)

Train dataframe columns:
['patient_id', 'series_id', 'image_id', 'laterality', 'view', 'height', 'width', 'breast_birads', 'breast_density', 'finding_categories', 'finding_birads', 'xmin', 'ymin', 'xmax', 'ymax', 'split', 'resized_xmin', 'resized_ymin', 'resized_xmax', 'resized_ymax', 'fold', 'Architectural_Distortion', 'Asymmetry', 'Focal_Asymmetry', 'Global_Asymmetry', 'Mass', 'Nipple_Retraction', 'No_Finding', 'Skin_Retraction', 'Skin_Thickening', 'Suspicious_Calcification', 'Suspicious_Lymph_Node', 'density']

Finding categories:
finding_categories
['No Finding']                                                                                      11689
['Mass']                                                                                              721
['Suspicious Calcification']                                                                          234
['Focal Asymmetry']                                                

In [34]:
from tqdm.auto import tqdm

In [47]:
NUM_EPOCHS = 10

best_val_auc = -float("inf")

CHECKPOINT_PATH = (
    "/content/MammoGraph/"
    "best_gnt_model.pth"
)

for epoch in range(NUM_EPOCHS):

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    train_loss = 0.0
    train_batches = 0

    for batch in tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"
    ):

        images = batch["images"].to(
            DEVICE,
            non_blocking=True
        )

        image_labels = batch[
            "image_labels"
        ].to(DEVICE)

        # Frozen CNN
        with torch.no_grad():

            feature_map = feature_extractor(
                images
            )

        nodes, coords = feature_map_to_nodes(
            feature_map
        )

        neighbors = build_spatial_knn_neighbors(
            coords,
            k=8
        )

        outputs = model(
            nodes,
            neighbors
        )

        objectness_targets = []
        node_class_targets = []
        bbox_targets = []

        for b in range(images.shape[0]):

            obj, node_cls, bbox = \
                build_gnt_targets(
                    coords[b],
                    batch["boxes"][b],
                    batch["box_labels"][b]
                )

            objectness_targets.append(obj)
            node_class_targets.append(node_cls)
            bbox_targets.append(bbox)

        objectness_targets = torch.stack(
            objectness_targets
        )

        node_class_targets = torch.stack(
            node_class_targets
        )

        bbox_targets = torch.stack(
            bbox_targets
        )

        loss, loss_dict = gnt_loss(
            outputs,
            image_labels,
            objectness_targets,
            node_class_targets,
            bbox_targets
        )

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss += loss.item()
        train_batches += 1

    train_loss /= max(train_batches, 1)

    print(
        f"\nEpoch {epoch+1}/{NUM_EPOCHS}"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    all_targets = []
    all_probs = []

    val_loss = 0.0
    val_batches = 0

    with torch.no_grad():

        for batch in tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"
        ):

            images = batch["images"].to(
                DEVICE,
                non_blocking=True
            )

            image_labels = batch[
                "image_labels"
            ].to(DEVICE)

            feature_map = feature_extractor(
                images
            )

            nodes, coords = feature_map_to_nodes(
                feature_map
            )

            neighbors = build_spatial_knn_neighbors(
                coords,
                k=8
            )

            outputs = model(
                nodes,
                neighbors
            )

            objectness_targets = []
            node_class_targets = []
            bbox_targets = []

            for b in range(images.shape[0]):

                obj, node_cls, bbox = \
                    build_gnt_targets(
                        coords[b],
                        batch["boxes"][b],
                        batch["box_labels"][b]
                    )

                objectness_targets.append(obj)
                node_class_targets.append(node_cls)
                bbox_targets.append(bbox)

            objectness_targets = torch.stack(
                objectness_targets
            )

            node_class_targets = torch.stack(
                node_class_targets
            )

            bbox_targets = torch.stack(
                bbox_targets
            )

            loss, _ = gnt_loss(
                outputs,
                image_labels,
                objectness_targets,
                node_class_targets,
                bbox_targets
            )

            val_loss += loss.item()
            val_batches += 1

            probs = torch.sigmoid(
                outputs["image_logits"]
            )

            all_targets.append(
                image_labels.cpu().numpy()
            )

            all_probs.append(
                probs.cpu().numpy()
            )

    val_loss /= max(val_batches, 1)

    all_targets = np.concatenate(
        all_targets,
        axis=0
    )

    all_probs = np.concatenate(
        all_probs,
        axis=0
    )

    class_aucs = []

    for c in range(NUM_CLASSES):

        # Skip classes with only one label value
        if len(np.unique(all_targets[:, c])) < 2:
            continue

        try:
            auc = roc_auc_score(
                all_targets[:, c],
                all_probs[:, c]
            )

            class_aucs.append(auc)

        except ValueError:
            pass

    macro_auc = (
        np.mean(class_aucs)
        if class_aucs
        else 0.0
    )

    scheduler.step(macro_auc)

    print(
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"Val Macro AUROC: {macro_auc:.4f}"
    )

    # ========================================================
    # CHECKPOINT
    # ========================================================

    if macro_auc > best_val_auc:

        best_val_auc = macro_auc

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_macro_auc": macro_auc,
                "finding_classes": FINDING_CLASSES,
            },
            CHECKPOINT_PATH
        )

        print(
            "✓ Best GNT checkpoint saved."
        )

Epoch 1/10 [Train]:   0%|          | 0/3200 [00:00<?, ?it/s]


Epoch 1/10
Train Loss: 0.7901


Epoch 1/10 [Val]:   0%|          | 0/800 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b5351c1ccc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b5351c1ccc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Val Loss: 0.3825
Val Macro AUROC: 0.6938
✓ Best GNT checkpoint saved.


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b5351c1ccc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b5351c1ccc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/


Epoch 10/10
Train Loss: 0.5592


Epoch 10/10 [Val]:   0%|          | 0/800 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b5351c1ccc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b5351c1ccc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Val Loss: 0.2397
Val Macro AUROC: 0.6990


In [9]:
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=gnt_collate_fn
)

print("Test loader ready.")

Test loader ready.


In [39]:
# ============================================================
# TEST EVALUATION — BEST GNT CHECKPOINT
# ============================================================

import torch
import numpy as np
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

CHECKPOINT_PATH = "/content/MammoGraph/best_gnt_model.pth"

# ------------------------------------------------------------
# Load best checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

print("=" * 60)
print("GNT TEST EVALUATION")
print("=" * 60)

print(
    f"Best checkpoint epoch: "
    f"{checkpoint['epoch']}"
)

print(
    f"Best validation Macro AUROC: "
    f"{checkpoint['val_macro_auc']:.4f}"
)

print()

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

all_targets = []
all_probs = []

with torch.no_grad():

    for batch in tqdm(
        test_loader,
        desc="Testing",
        dynamic_ncols=True
    ):

        images = batch["images"].to(
            DEVICE,
            non_blocking=True
        )

        image_labels = batch[
            "image_labels"
        ].to(DEVICE)

        # ----------------------------------------------------
        # Frozen CNN feature extractor
        # ----------------------------------------------------

        feature_map = feature_extractor(
            images
        )

        # ----------------------------------------------------
        # Convert feature map → graph nodes
        # ----------------------------------------------------

        nodes, coords = feature_map_to_nodes(
            feature_map
        )

        # ----------------------------------------------------
        # Build same 8-NN spatial graph used in training
        # ----------------------------------------------------

        neighbors = build_spatial_knn_neighbors(
            coords,
            k=8
        )

        # ----------------------------------------------------
        # GNT forward pass
        # ----------------------------------------------------

        outputs = model(
            nodes,
            neighbors
        )

        # ----------------------------------------------------
        # Image-level predictions
        # ----------------------------------------------------

        probs = torch.sigmoid(
            outputs["image_logits"]
        )

        all_targets.append(
            image_labels.cpu().numpy()
        )

        all_probs.append(
            probs.cpu().numpy()
        )

# ------------------------------------------------------------
# Combine all test batches
# ------------------------------------------------------------

all_targets = np.concatenate(
    all_targets,
    axis=0
)

all_probs = np.concatenate(
    all_probs,
    axis=0
)

print(
    f"\nTest samples: "
    f"{len(all_targets)}"
)

print(
    f"Number of classes: "
    f"{all_targets.shape[1]}"
)

# ------------------------------------------------------------
# Per-class AUROC
# ------------------------------------------------------------

class_aurocs = {}

print("\n" + "=" * 60)
print("PER-CLASS TEST AUROC")
print("=" * 60)

for i, class_name in enumerate(
    FINDING_CLASSES
):

    y_true = all_targets[:, i]
    y_score = all_probs[:, i]

    if len(np.unique(y_true)) < 2:

        class_aurocs[class_name] = np.nan

        print(
            f"{class_name:<30} N/A"
        )

        continue

    auc = roc_auc_score(
        y_true,
        y_score
    )

    class_aurocs[class_name] = auc

    print(
        f"{class_name:<30} "
        f"{auc:.4f}"
    )

# ------------------------------------------------------------
# Macro AUROC
# ------------------------------------------------------------

valid_aurocs = [
    auc
    for auc in class_aurocs.values()
    if not np.isnan(auc)
]

test_macro_auc = np.mean(
    valid_aurocs
)

print("\n" + "=" * 60)

print(
    f"TEST MACRO AUROC: "
    f"{test_macro_auc:.4f}"
)

print("=" * 60)

GNT TEST EVALUATION
Best checkpoint epoch: 7
Best validation Macro AUROC: 0.7216



Testing:   0%|          | 0/1000 [00:00<?, ?it/s]


Test samples: 4000
Number of classes: 10

PER-CLASS TEST AUROC
Mass                           0.6222
Suspicious Calcification       0.8513
Focal Asymmetry                0.5904
Architectural Distortion       0.6113
Asymmetry                      0.6076
Suspicious Lymph Node          0.9288
Skin Thickening                0.6768
Global Asymmetry               0.6989
Nipple Retraction              0.7184
Skin Retraction                0.5889

TEST MACRO AUROC: 0.6895


In [40]:
# ============================================================
# GNT — COMPLETE TEST METRICS
# ============================================================

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

THRESHOLD = 0.5

# ------------------------------------------------------------
# Convert probabilities to binary predictions
# ------------------------------------------------------------

gnt_preds = (all_probs >= THRESHOLD).astype(int)

# ------------------------------------------------------------
# Per-class metrics
# ------------------------------------------------------------

print("=" * 120)
print("GNT TEST METRICS")
print("=" * 120)

print(f"Threshold: {THRESHOLD}")
print(f"Test samples: {len(all_targets)}")
print(f"Number of classes: {all_targets.shape[1]}")
print()

header = (
    f"{'Class':<30}"
    f"{'Acc':>9}"
    f"{'BalAcc':>9}"
    f"{'Precision':>11}"
    f"{'Recall':>9}"
    f"{'Specificity':>14}"
    f"{'F1':>9}"
    f"{'AUROC':>9}"
    f"{'AUPRC':>9}"
)

print(header)
print("-" * 120)

per_class_metrics = {}

for i, class_name in enumerate(FINDING_CLASSES):

    y_true = all_targets[:, i]
    y_pred = gnt_preds[:, i]
    y_score = all_probs[:, i]

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    # Metrics
    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    balanced_acc = balanced_accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    # AUROC
    if len(np.unique(y_true)) >= 2:
        auroc = roc_auc_score(
            y_true,
            y_score
        )

        auprc = average_precision_score(
            y_true,
            y_score
        )
    else:
        auroc = np.nan
        auprc = np.nan

    per_class_metrics[class_name] = {
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "Accuracy": accuracy,
        "Balanced Accuracy": balanced_acc,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1": f1,
        "AUROC": auroc,
        "AUPRC": auprc
    }

    print(
        f"{class_name:<30}"
        f"{accuracy:>9.4f}"
        f"{balanced_acc:>9.4f}"
        f"{precision:>11.4f}"
        f"{recall:>9.4f}"
        f"{specificity:>14.4f}"
        f"{f1:>9.4f}"
        f"{auroc:>9.4f}"
        f"{auprc:>9.4f}"
    )


# ============================================================
# MACRO METRICS
# ============================================================

macro_accuracy = np.mean([
    x["Accuracy"]
    for x in per_class_metrics.values()
])

macro_balanced_accuracy = np.mean([
    x["Balanced Accuracy"]
    for x in per_class_metrics.values()
])

macro_precision = np.mean([
    x["Precision"]
    for x in per_class_metrics.values()
])

macro_recall = np.mean([
    x["Recall"]
    for x in per_class_metrics.values()
])

macro_specificity = np.mean([
    x["Specificity"]
    for x in per_class_metrics.values()
])

macro_f1 = np.mean([
    x["F1"]
    for x in per_class_metrics.values()
])

macro_auroc = np.mean([
    x["AUROC"]
    for x in per_class_metrics.values()
    if not np.isnan(x["AUROC"])
])

macro_auprc = np.mean([
    x["AUPRC"]
    for x in per_class_metrics.values()
    if not np.isnan(x["AUPRC"])
])


# ============================================================
# MICRO METRICS
# ============================================================

micro_accuracy = accuracy_score(
    all_targets.flatten(),
    gnt_preds.flatten()
)

micro_precision = precision_score(
    all_targets.flatten(),
    gnt_preds.flatten(),
    zero_division=0
)

micro_recall = recall_score(
    all_targets.flatten(),
    gnt_preds.flatten(),
    zero_division=0
)

micro_f1 = f1_score(
    all_targets.flatten(),
    gnt_preds.flatten(),
    zero_division=0
)

micro_auroc = roc_auc_score(
    all_targets.flatten(),
    all_probs.flatten()
)

micro_auprc = average_precision_score(
    all_targets.flatten(),
    all_probs.flatten()
)


# ============================================================
# OVERALL CONFUSION COUNTS
# ============================================================

total_tp = sum(
    x["TP"] for x in per_class_metrics.values()
)

total_fp = sum(
    x["FP"] for x in per_class_metrics.values()
)

total_tn = sum(
    x["TN"] for x in per_class_metrics.values()
)

total_fn = sum(
    x["FN"] for x in per_class_metrics.values()
)


# ============================================================
# PRINT OVERALL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("OVERALL GNT TEST METRICS")
print("=" * 70)

print(f"Threshold                       : {THRESHOLD}")

print("\nCONFUSION COUNTS")
print("-" * 70)

print(f"Total TP                        : {total_tp}")
print(f"Total FP                        : {total_fp}")
print(f"Total TN                        : {total_tn}")
print(f"Total FN                        : {total_fn}")

print("\nMACRO METRICS")
print("-" * 70)

print(f"Macro Accuracy                  : {macro_accuracy:.6f}")
print(f"Macro Balanced Accuracy         : {macro_balanced_accuracy:.6f}")
print(f"Macro Precision                : {macro_precision:.6f}")
print(f"Macro Recall / Sensitivity      : {macro_recall:.6f}")
print(f"Macro Specificity               : {macro_specificity:.6f}")
print(f"Macro F1                        : {macro_f1:.6f}")
print(f"Macro AUROC                     : {macro_auroc:.6f}")
print(f"Macro AUPRC                     : {macro_auprc:.6f}")

print("\nMICRO METRICS")
print("-" * 70)

print(f"Micro Accuracy                  : {micro_accuracy:.6f}")
print(f"Micro Precision                 : {micro_precision:.6f}")
print(f"Micro Recall / Sensitivity      : {micro_recall:.6f}")
print(f"Micro F1                        : {micro_f1:.6f}")
print(f"Micro AUROC                     : {micro_auroc:.6f}")
print(f"Micro AUPRC                     : {micro_auprc:.6f}")

print("=" * 70)


# ============================================================
# CONFUSION COUNTS PER CLASS
# ============================================================

print("\n")
print("=" * 70)
print("GNT CONFUSION COUNTS PER CLASS")
print("=" * 70)

for class_name, m in per_class_metrics.items():

    print(f"\n{class_name}")
    print(f"TP: {m['TP']}")
    print(f"FP: {m['FP']}")
    print(f"TN: {m['TN']}")
    print(f"FN: {m['FN']}")

GNT TEST METRICS
Threshold: 0.5
Test samples: 4000
Number of classes: 10

Class                               Acc   BalAcc  Precision   Recall   Specificity       F1    AUROC    AUPRC
------------------------------------------------------------------------------------------------------------------------
Mass                             0.9247   0.5623     0.2267   0.1553        0.9693   0.1843   0.6222   0.1318
Suspicious Calcification         0.9758   0.6122     0.6000   0.2286        0.9959   0.3310   0.8513   0.3654
Focal Asymmetry                  0.9865   0.4997     0.0000   0.0000        0.9995   0.0000   0.5904   0.0293
Architectural Distortion         0.9940   0.5000     0.0000   0.0000        1.0000   0.0000   0.6113   0.0259
Asymmetry                        0.9950   0.5000     0.0000   0.0000        1.0000   0.0000   0.6076   0.0067
Suspicious Lymph Node            0.9972   0.4999     0.0000   0.0000        0.9997   0.0000   0.9288   0.1654
Skin Thickening                  0.